# Intermediate 03 - 인증 (Authentication)

이 노트북에서는:
- SIP 인증의 기본 원리를 배웁니다
- Digest 인증 흐름을 이해합니다
- `www_authorize`, `proxy_authorize` 사용법을 익힙니다
- 인증 실패 시 응답 처리를 연습합니다

---

## SIP 인증이란?

SIP 인증은 **HTTP Digest 인증**과 동일한 방식입니다:

```
1. UA → Proxy: INVITE (인증 없이)
2. Proxy → UA: 407 Proxy Authentication Required
3. UA → Proxy: INVITE (credentials 포함)
4. Proxy → UA: 인증 성공 → 요청 처리
```

두 가지 인증 타입:
- **WWW-Authenticate**: Registrar가 UA에게 요청 (`401 Unauthorized`)
- **Proxy-Authenticate**: Proxy가 UA에게 요청 (`407 Proxy Auth Required`)

실무에서는 대부분 **Proxy 인증(407)**을 사용합니다.

## 1. REGISTER 요청에 인증 적용

가장 흔한 인증 시나리오입니다. REGISTER 시 credentials를 확인합니다.

In [ ]:
%%sip REGISTER
From: <sip:1001@example.com>;tag=reg1
To: <sip:1001@example.com>

In [ ]:
# 인증이 없는 REGISTER — 401로 거부
$var(has_auth) = "no";

if ($var(has_auth) == "no") {
    xlog("No credentials — challenging REGISTER");
    send_reply(401, "Unauthorized");
} else {
    xlog("Credentials verified — saving registration");
}

## 2. INVITE에 Proxy 인증 적용

통화 요청 시 Proxy 인증(407)을 요구하는 패턴입니다.

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=inv1
To: <sip:1002@example.com>

In [ ]:
# 인증 상태 시뮬레이션
$var(authenticated) = "no";

if ($var(authenticated) != "yes") {
    xlog("No Proxy-Authorization header — sending 407");
    send_reply(407, "Proxy Authentication Required");
} else {
    xlog("Authenticated — proceeding with routing");
}

## 3. Digest 인증 흐름 시뮬레이션

실제 인증은 DB 조회가 필요하지만, 이 커널에서는 시뮬레이션으로 흐름을 이해합니다.

실제 Kamailio 코드:
```kamailio
# 실무 코드
if (!www_authorize("example.com", "subscriber")) {
    www_challenge("example.com", "1");
    exit;
}
```

In [ ]:
# 1차 요청: 인증 없음
%%sip INVITE
From: <sip:1001@example.com>;tag=first
To: <sip:2001@example.com>

In [ ]:
# 인증 확인 — 헤더가 없으므로 407 응답
xlog("Step 1: UA sends INVITE without credentials");
send_reply(407, "Proxy Authentication Required");
xlog("Step 2: Proxy responds with 407 + challenge");

In [ ]:
# 2차 요청: UA가 credentials 포함하여 재전송
%%sip INVITE
From: <sip:1001@example.com>;tag=first
To: <sip:2001@example.com>

In [ ]:
# 인증 성공 시뮬레이션
xlog("Step 3: UA resends INVITE with credentials");
$var(auth_ok) = "yes";

if ($var(auth_ok) == "yes") {
    xlog("Step 4: Authentication successful");
    xlog("Proceeding to route call to $ru");
} else {
    send_reply(403, "Forbidden");
}

## 4. 인증 실패 처리

비밀번호가 틀리거나 계정이 없는 경우의 처리입니다.

실무 코드:
```kamailio
if (!proxy_authorize("", "subscriber")) {
    proxy_challenge("", "1");
    exit;
}
if (!db_check_from()) {
    sl_send_reply("403", "Forbidden");
    exit;
}
consume_credentials();  # 보안: 인증 헤더 제거
```

In [ ]:
# 잘못된 credentials 시뮬레이션
$var(auth_result) = "fail";

if ($var(auth_result) == "fail") {
    xlog("Authentication FAILED — wrong password or unknown user");
    send_reply(403, "Forbidden");
} else {
    xlog("Authentication OK");
}

## 5. 메서드별 인증 정책

실무에서는 메서드마다 인증 정책이 다릅니다:

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=p1
To: <sip:2001@example.com>

In [ ]:
# 메서드별 인증 정책 시뮬레이션
$var(authenticated) = "yes";

if (is_method("REGISTER")) {
    xlog("REGISTER: requires www_authorize");
    if ($var(authenticated) != "yes") {
        send_reply(401, "Unauthorized");
    } else {
        xlog("REGISTER authenticated — saving location");
    }
} else {
    if (is_method("INVITE")) {
        xlog("INVITE: requires proxy_authorize");
        if ($var(authenticated) != "yes") {
            send_reply(407, "Proxy Authentication Required");
        } else {
            xlog("INVITE authenticated — routing call");
        }
    } else {
        xlog("Other method: no auth required");
    }
}

## 6. consume_credentials() — 보안 모범 사례

인증 성공 후 `consume_credentials()`를 호출하면 Authorization 헤더가 제거됩니다.
이렇게 하면 다음 홉으로 인증 정보가 유출되지 않습니다.

```kamailio
proxy_authorize("", "subscriber");
consume_credentials();  # Authorization 헤더 제거
t_relay();              # 안전하게 전달
```

In [ ]:
# 인증 후 헤더 정리 시뮬레이션
$var(user) = $(fu{uri.user});
xlog("Authenticated user: $var(user)");
xlog("Consuming credentials — removing auth headers");
xlog("Safe to relay: $ru");

---

### 요약

| 개념 | 설명 |
|------|------|
| WWW 인증 (401) | Registrar → UA, REGISTER에 사용 |
| Proxy 인증 (407) | Proxy → UA, INVITE 등에 사용 |
| www_authorize | DB에서 credentials 확인 |
| proxy_authorize | Proxy 인증 확인 |
| consume_credentials | 인증 헤더 제거 (보안) |
| 인증 실패 | 401/403/407 응답 |

다음 노트북: **Intermediate/04-registration-and-location.ipynb** →